<a href="https://colab.research.google.com/github/duartedani/tbh-esportes-estudos/blob/main/02_06_ETL_extracao_base_TBH_Esportes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



#### 1º - Instalação de Bibliotecas

 **boto3**
- É o SDK oficial da AWS para Python
- Serve para acessar serviços da Amazon:
  - S3
  - Lambda
  - DynamoDB
  - Entre outros

 **camelot-py**
- Biblioteca especializada em extração de tabelas de PDFs
- Principais funcionalidades:
  - Leitura de arquivos PDF
  - Detecção automática de tabelas
  - Conversão para DataFrame (formato semelhante a Excel/CSV)

 **pandas**
- Biblioteca padrão para análise de dados em Python
- Utilizada para:
  - Manipulação de tabelas
  - Tratamento de dados
  - Exportação para CSV, Excel e outros formatos


In [ ]:
!pip install boto3 camelot-py pandas
!pip install awscli


In [ ]:
!aws --version

In [ ]:
!aws configure

In [ ]:
!aws sts get-caller-identity


#### 2º - Importação das bibliotecas

    import boto3
        - baixar PDF de um bucket S3
        - listar arquivos na nuvem
        - enviar CSV processado de volta

    import camelot
        - ler PDFs com tabelas
        - extrair dados estruturados (tipo planilha)
        - converter tabelas do PDF em DataFrame
    
    import pandas as pd
        - manipular tabelas (DataFrame)
        - limpar dados
        - salvar CSV

    import re
        - biblioteca de expressões regulares
        - limpar textos
        - remover caracteres estranhos do PDF
        - extrair padrões (ex: números, datas, nomes)

    import os
        - trabalha em pastas e arquivos
        - criar diretórios
        - verificar se arquivos existem
        - trabalhar com caminhos

    

In [ ]:
import boto3
import camelot
from tqdm import tqdm
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

import re
import os
import io

#### 3º - Conexão com AWS S3

    - Define as credenciais de autenticação
    - S3 → objeto que fala com a AWS
    - bucket → pasta onde estão os PDFs
    Colocando as credencias:

        Será solicitado:
            - access_key
            - secret_key
            - region
            - output format

In [ ]:
!aws configure list

In [ ]:
s3 = boto3.client("s3")

In [ ]:
s3 = boto3.client("s3")

response = s3.list_buckets()
print(response["Buckets"])

In [ ]:
s3.list_objects_v2(Bucket="tbh-resultados-raw")

#### 4º - Transformar o nome do arquivo em dados estruturados

        - Obtem os metadados da corrida e que serão adicionados na estrutura da tabela posteriormente.
        - Dados obtidos:
             - nome da corrida,
             - ano
             - genero
             - distancia da prova


In [ ]:
def extrair_metadados(key):
    filename = os.path.basename(key).replace('.pdf', '')

    # 📅 Data
    data_match = re.search(r'\d{4}-\d{2}-\d{2}', filename)
    data = data_match.group(0) if data_match else None

    # 🏃 Corrida (entre data e ano)
    corrida_match = re.search(r'\d{4}-\d{2}-\d{2}_(.+?)_\d{4}', filename)
    corrida = corrida_match.group(1) if corrida_match else None

    # 🚻 + 📏 Distância (dois padrões possíveis)

    # Padrão 1: 06KM-FEMININO
    pattern1 = r'(?P<dist>\d+(?:\.\d+)?)KM-(?P<gen>FEMININO|MASCULINO)'

    # Padrão 2: FEMININO-12.8KM
    pattern2 = r'(?P<gen>FEMININO|MASCULINO)-(?P<dist>\d+(?:\.\d+)?)KM'

    match = re.search(pattern1, filename, re.IGNORECASE)

    if not match:
        match = re.search(pattern2, filename, re.IGNORECASE)

    if match:
        genero = match.group('gen').upper()
        distancia = float(match.group('dist'))
    else:
        genero = None
        distancia = None

    return {
        'data': data,
        'corrida': corrida,
        'genero': genero,
        'distancia_km': distancia
    }

##### GUIA : Identificar o nome nome dos arquivos no S3


In [ ]:
bucket = "tbh-resultados-raw"
prefix = "TBH Esportes/corridas_2026_completo_final/"

In [ ]:
def listar_arquivos_s3(s3, bucket, prefixo=None):
    arquivos = []

    kwargs = {'Bucket': bucket}
    if prefixo:
        kwargs['Prefix'] = prefixo

    while True:
        response = s3.list_objects_v2(**kwargs)

        for obj in response.get('Contents', []):
            arquivos.append(obj['Key'])

        # verifica se tem mais páginas
        if response.get('IsTruncated'):
            kwargs['ContinuationToken'] = response['NextContinuationToken']
        else:
            break

    return arquivos

In [ ]:
keys = listar_arquivos_s3(s3, bucket)

print("Total de arquivos:", len(keys))

print("\nPrimeiros 5 arquivos:\n")
for i, k in enumerate(keys[:5], 1):
    print(f"{i} - {k}")

##### EXEMPLO : Obter o nome de 1 arquivo para validação

In [ ]:
key = "TBH Esportes/corridas_2026_completo_final/2026-01-25_Park_Run_2026_09KM-FEMININO"

###### VALIDAÇÃO : Verificar como os dados referente a corrida estão retornando

In [ ]:
extrair_metadados(key)

#### 5º - Coleta os arquivos do S3 e retorna apenas o que é PDF

  
    - Percorre todos os arquivos do bucket/pasta no S3
    - Lida com paginação (caso tenha muitos arquivos)
    - Filtra apenas PDFs
    - Guarda tudo numa lista - pdf_keys



In [ ]:
paginator = s3.get_paginator('list_objects_v2')

pdf_keys = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    contents = page.get('Contents', [])

    for obj in contents:
        key = obj['Key']

        # garante que só pega PDFs reais (evita zip etc.)
        if key.lower().endswith('.pdf'):
            pdf_keys.append(key)

print(f"Total de PDFs encontrados: {len(pdf_keys)}")

#### 6º - Normalização dos cabeçalhos

In [ ]:
arquivos_pdf = []

for key in pdf_keys:
    obj = s3.get_object(Bucket=bucket, Key=key)
    pdf_bytes = obj['Body'].read()

    arquivos_pdf.append({
        "key": key,
        "pdf_bytes": pdf_bytes
    })

print("Arquivos carregados:", len(arquivos_pdf))

In [ ]:
from tqdm import tqdm
cabecalhos = []

for arq in tqdm(arquivos_pdf, desc="Processando PDFs"):
    try:
        obj = s3.get_object(Bucket=bucket, Key=arq["key"])
        pdf_bytes = obj["Body"].read()

        tables = camelot.read_pdf(io.BytesIO(pdf_bytes))

        if len(tables) == 0:
            continue

        df = tables[0].df
        header = tuple(df.iloc[0].astype(str).str.strip().tolist())

        cabecalhos.append({
            "key": arq["key"],
            "cabecalho": header
        })

    except Exception as e:
        print("Erro no arquivo:", arq["key"], e)


In [ ]:
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

In [ ]:

#s3 = boto3.client('s3')
#bucket = "tbh-resultados-raw"
#prefix = "TBH Esportes/corridas_2026_completo_final/"

def listar_arquivos_s3(s3_client, bucket_name, prefixo=None):
    arquivos = []
    kwargs = {'Bucket': bucket_name}
    if prefixo:
        kwargs['Prefix'] = prefixo

    while True:
        response = s3_client.list_objects_v2(**kwargs)
        for obj in response.get('Contents', []):
            arquivos.append(obj['Key'])
        if response.get('IsTruncated'):
            kwargs['ContinuationToken'] = response['NextContinuationToken']
        else:
            break
    return arquivos

pdf_keys = listar_arquivos_s3(s3, bucket, prefix)

cabecalhos = []
not_processed_pdfs = []

for key in tqdm(pdf_keys, desc="Processando PDFs"):
    try:
        obj = s3.get_object(Bucket=bucket, Key=key)
        pdf_bytes = obj['Body'].read()

        tables = camelot.read_pdf(io.BytesIO(pdf_bytes))

        if len(tables) == 0:
            print(f"Aviso: Nenhum cabeçalho encontrado no PDF (sem tabelas): {key}")
            not_processed_pdfs.append(key)
            continue

        df = tables[0].df
        header = tuple(df.iloc[0].astype(str).str.strip().tolist())

        cabecalhos.append({
            "key": key,
            "cabecalho": header
        })

    except Exception as e:
        print(f"Erro ao processar o arquivo: {key} - {e}")
        not_processed_pdfs.append(key)

print(f"Total de PDFs no S3: {len(pdf_keys)}")
print(f"Cabeçalhos coletados: {len(cabecalhos)}")
print(f"PDFs não processados: {len(not_processed_pdfs)}")

if not_processed_pdfs:
    print("Lista de PDFs não processados:")
    for pdf in not_processed_pdfs:
        print(pdf)

from collections import Counter

contagem = Counter([item["cabecalho"] for item in cabecalhos])

print("Tipos de cabeçalho:", len(contagem))

for i, (cab, qtd) in enumerate(contagem.items()):
    print(f"\n=== Tipo {i+1} ({qtd} arquivos) ===")
    print(cab)




In [ ]:
# confirmar a variacao de cabeçalhos
len(set(cabecalhos[i]["cabecalho"] for i in range(len(cabecalhos))))

In [ ]:
# Listar exemplos para cada tipo de cabeçalho
for i, (cab, qtd) in enumerate(contagem.items()):
    print(f"\n=== Tipo {i+1} ({qtd} arquivos) ===")
    print(f"Cabeçalho: {cab}")

    # Encontrar os primeiros 3 arquivos deste tipo
    exemplos = [item["key"] for item in cabecalhos if item["cabecalho"] == cab][:3]
    for ex in exemplos:
        print(f"  - {ex}")


In [ ]:
#08/05
standard_header = (
    'Coloc.', 'Num.', 'Nome', 'Sx.', 'Idd.', 'Faixa', 'Cl.Fx.', 'Equipe', 'Tempo', 'Liquido'
)

In [ ]:
#08/05
current_header_tipo2 = (
    'Coloc.', 'Numero', 'Nome', 'Id.', 'Fx.Et.', 'Cl.Fx.', 'Sx.', 'C.', 'Equipe', 'Tempo'
)

In [ ]:
#14/05
def is_tipo2(df):
    primeira_linha = df.iloc[0].astype(str).str.upper().tolist()

    palavras_chave = ['COLOC', 'NUM', 'NOME']

    return all(any(p in col for col in primeira_linha) for p in palavras_chave)

In [ ]:
#14/05
def normalize_tipo2(df_bruto):
    df = df_bruto.copy()

    # Remove cabeçalho
    df = df.iloc[1:].reset_index(drop=True)

    normalized_df = pd.DataFrame()

    normalized_df['Coloc.'] = df[0]
    normalized_df['Num.']   = df[1]
    normalized_df['Nome']   = df[2]
    normalized_df['Sx.']    = df[3]
    normalized_df['Idd.']   = df[4]
    normalized_df['Faixa']  = df[5]
    normalized_df['Cl.Fx.'] = df[6]
    normalized_df['Equipe'] = df[7]
    normalized_df['Tempo']  = df[8]
    normalized_df['Liquido'] = df[9]

    # Limpeza
    normalized_df['Nome'] = (
        normalized_df['Nome']
        .astype(str)
        .str.replace('\n', ' ', regex=False)
        .str.strip()
    )

    return normalized_df


In [ ]:
#14/05
def processar_pdf(key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    pdf_bytes = obj["Body"].read()

    tables = camelot.read_pdf(io.BytesIO(pdf_bytes), flavor='lattice')

    if len(tables) == 0:
        tables = camelot.read_pdf(io.BytesIO(pdf_bytes), flavor='stream')

    if len(tables) == 0:
        raise Exception("Nenhuma tabela encontrada")

    df_bruto = tables[0].df

    # 🔥 Detecta tipo automaticamente
    if is_tipo2(df_bruto):
        df_final = normalize_tipo2(df_bruto)
    else:
        raise Exception("Não é tipo 2")

    return df_final

In [ ]:
resultados = []
erros = []

total = len(pdf_keys)

for i, key in enumerate(pdf_keys, start=1):
    if i % 5 == 0 or i == total:
        print(f"📄 {i}/{total}")

    try:
        df = processar_pdf(key)
        df["arquivo"] = key
        resultados.append(df)

    except Exception as e:
        erros.append({
            "arquivo": key,
            "erro": str(e)
        })

print("\n✅ Processamento finalizado!")
print(f"✔ Sucesso: {len(resultados)} arquivos")
print(f"❌ Erros: {len(erros)} arquivos")

In [ ]:
df_erros = pd.DataFrame(erros)
display(df_erros)

In [ ]:
#18/05
def remover_cabecalho_duplicado(df):
    return df[df[0] != 'Coloc.'].reset_index(drop=True)

In [ ]:
#18/05
key_teste = "TBH Esportes/corridas_2026_completo_final/2026-01-25_Park_Run_2026_06KM-FEMININO-1.pdf"

def processar_tipo1_teste(key, bucket):
    print(f"📄 Processando arquivo:\n{key}\n")

    obj = s3.get_object(Bucket=bucket, Key=key)
    pdf_bytes = obj["Body"].read()

    # 🔥 Lê todas as páginas
    tables = camelot.read_pdf(io.BytesIO(pdf_bytes), pages='all', flavor='lattice')

    if len(tables) == 0:
        print("⚠️ Tentando com stream...")
        tables = camelot.read_pdf(io.BytesIO(pdf_bytes), pages='all', flavor='stream')

    if len(tables) == 0:
        raise Exception("❌ Nenhuma tabela encontrada")

    print(f"✅ Tabelas encontradas: {len(tables)}\n")

    dfs = []

    for i, table in enumerate(tables):
        df = table.df
        print(f"➡️ Página {i+1} - shape: {df.shape}")

        # 🔥 Remove cabeçalho duplicado
        df = df[df[0] != 'Coloc.']

        dfs.append(df)

    # 🔗 Junta tudo
    df_full = pd.concat(dfs, ignore_index=True)

    print("\n🧾 Preview bruto consolidado:")
    display(df_full.head())

    # Remove primeira linha (cabeçalho original)
    df_full = df_full.iloc[1:].reset_index(drop=True)

    # 🔥 Normalização por posição
    df_final = pd.DataFrame()

    df_final['Coloc.'] = df_full[0]
    df_final['Num.']   = df_full[1]
    df_final['Nome']   = df_full[2]
    df_final['Sx.']    = df_full[3]
    df_final['Idd.']   = df_full[4]
    df_final['Faixa']  = df_full[5]
    df_final['Cl.Fx.'] = df_full[6]
    df_final['Equipe'] = df_full[7]
    df_final['Tempo']  = df_full[8]
    df_final['Liquido'] = df_full[9]

    # 🧹 Limpeza
    for col in df_final.columns:
        df_final[col] = (
            df_final[col]
            .astype(str)
            .str.replace('\n', ' ', regex=False)
            .str.strip()
        )

    # ❌ Remove linhas vazias
    df_final = df_final[df_final['Nome'] != '']

    # 🧠 METADADOS
    df_final['arquivo'] = key

      # Data
    match = re.search(r'\d{4}-\d{2}-\d{2}', key)
    df_final['data_evento'] = match.group(0) if match else None

    # Nome da prova
    nome_arquivo = key.split("/")[-1].replace(".pdf", "")
    df_final['nome_prova'] = nome_arquivo

    # Distância (ex: 06KM)
    match_km = re.search(r'(\d{2})KM', key.upper())
    df_final['distancia_km'] = match_km.group(1) if match_km else None

    # Categoria (MASCULINO / FEMININO)
    if "FEMININO" in key.upper():
        df_final['categoria'] = "FEMININO"
    elif "MASCULINO" in key.upper():
        df_final['categoria'] = "MASCULINO"
    else:
        df_final['categoria'] = None

    print("\n🧾 Preview final:")
    display(df_final.head())

    print("\n📊 Linhas finais:", len(df_final))

    return df_final

In [ ]:
#20/05
import io
import re
import boto3
import pandas as pd
import numpy as np
import camelot

S3_BUCKET_RAW = 'tbh-resultados-raw'
S3_BUCKET_SILVER = 'tbh-resultados-silver'

s3_client = boto3.client('s3')

# Cabeçalho padrão
STANDARD_HEADER = ['Coloc.', 'Num.', 'Nome', 'Sx.', 'Idd.', 'Faixa', 'Cl.Fx.', 'Equipe', 'Tempo', 'Liquido']

def extrair_metadados_from_key(key: str):
    filename = os.path.basename(key).replace('.pdf', '')

    data_match = re.search(r'\d{4}-\d{2}-\d{2}', filename)
    data = data_match.group(0) if data_match else None

    corrida_match = re.search(r'\d{4}-\d{2}-\d{2}_(.+?)_\d{4}', filename)
    corrida = corrida_match.group(1) if corrida_match else None

    # Padrões de genero + distância
    pattern1 = r'(?P<dist>\d+(?:\.\d+)?)KM-(?P<gen>FEMININO|MASCULINO)'
    pattern2 = r'(?P<gen>FEMININO|MASCULINO)-(?P<dist>\d+(?:\.\d+)?)KM'

    m = re.search(pattern1, filename, re.IGNORECASE) or re.search(pattern2, filename, re.IGNORECASE)
    if m:
        genero = m.group('gen').upper()
        distancia = float(m.group('dist'))
    else:
        genero = None
        distancia = None

    return {
        'data': data,
        'corrida': corrida,
        'genero': genero,
        'distancia_km': distancia,
        'arquivo': filename + '.pdf',
    }

def is_tipo2(df_raw: pd.DataFrame) -> bool:
    primeira_linha = df_raw.iloc[0].astype(str).str.upper().tolist()
    palavras_chave = ['COLOC', 'NUM', 'NOME']
    return all(any(p in col for col in primeira_linha) for p in palavras_chave)

def header_tuple(df_raw: pd.DataFrame):
    return tuple(df_raw.iloc[0].astype(str).str.strip().tolist())



def normalize_tipo1(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza o DataFrame mantendo a primeira linha de cabeçalho
    e removendo apenas as repetições subsequentes.
    """
    if df_raw.empty:
        return df_raw

    df = df_raw.copy()

    # Padroniza as colunas para índices numéricos para evitar erros de referência
    df.columns = range(df.shape[1])

    # Lógica Sênior: Criamos uma máscara que mantém a linha 0
    # OU qualquer linha onde a coluna 2 não seja o cabeçalho duplicado 'Nome'
    mask = (df.index == df.index[0]) | (df[2] != 'Nome')

    return df[mask].reset_index(drop=True)



    # Mapear colunas
    out = pd.DataFrame()
    out['Coloc.']  = df[0]
    out['Num.']    = df[1]
    out['Nome']    = df[2]
    out['Sx.']     = df[6]
    out['Idd.']    = df[3]
    out['Faixa']   = df[4]
    out['Cl.Fx.']  = df[5]
    out['Equipe']  = df[8]
    out['Tempo']   = df[9]
    out['Liquido'] = ''

    # Limpeza
    out = out.apply(lambda s: s.astype(str).str.replace('\n', ' ', regex=False).str.strip())

    # Remove linhas inválidas
    out = out[out['Nome'] != '']

    return out

    out = out.apply(lambda s: s.astype(str).str.replace('\n', ' ', regex=False).str.strip())
    # remove linhas sem nome
    out = out[out['Nome'] != '']
    return out

def normalize_tipo2(df_raw: pd.DataFrame) -> pd.DataFrame:
    # Remove a primeira linha (cabeçalho original)
    df = df_raw.copy().iloc[1:].reset_index(drop=True)

    # Mapear colunas do Tipo 2 para padrão
    # Cabeçalho tipo 2 observado:
    # ('Coloc.', 'Numero', 'Nome', 'Id.', 'Fx.Et.', 'Cl.Fx.', 'Sx.', 'C.', 'Equipe', 'Tempo')
    out = pd.DataFrame()
    out['Coloc.']  = df[0]
    out['Num.']    = df[1]  # 'Numero' -> 'Num.'
    out['Nome']    = df[2]
    out['Sx.']     = df[6]  # 'Sx.' está na 6 no tipo2
    out['Idd.']    = df[3]  # 'Id.' -> 'Idd.'
    out['Faixa']   = df[4]  # 'Fx.Et.' -> 'Faixa'
    out['Cl.Fx.']  = df[5]
    out['Equipe']  = df[8]
    out['Tempo']   = df[9]
    out['Liquido'] = ''     # tipo 2 não tem 'Liquido' -> preencher vazio

    out = out.apply(lambda s: s.astype(str).str.replace('\n', ' ', regex=False).str.strip())
    out = out[out['Nome'] != '']
    return out

def add_metadados(df_padrao: pd.DataFrame, key: str) -> pd.DataFrame:
    meta = extrair_metadados_from_key(key)
    for k, v in meta.items():
        df_padrao[k] = v
    return df_padrao

def coerce_types(df: pd.DataFrame) -> pd.DataFrame:
    # Colunas numéricas básicas quando possível
    for col in ['Coloc.', 'Num.', 'Idd.']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    # Mantém Tempo/Liquido como string (podem ser hh:mm:ss)
    return df

def salvar_parquet_s3(df: pd.DataFrame, bucket: str, key_out: str):
    # Usa buffer em memória e faz upload via boto3
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    s3_client.put_object(Bucket=bucket, Key=key_out, Body=buf.getvalue(), ContentType='application/octet-stream')


In [ ]:
#02/06

def fix_shifted_rows(df):
    """
    Corrige o problema de colunas fundidas (ex: '1 1639') comum em extrações de PDF.
    """
    new_rows = []
    for _, row in df.iterrows():
        row_list = [str(v).strip() if pd.notna(v) else "" for v in row]
        # Se a primeira coluna tem dois números separados por espaço, separa-os
        if re.match(r'^\d+\s+\d+$', row_list[0]):
            parts = row_list[0].split()
            row_list = [parts[0], parts[1]] + row_list[1:-1]
        # Se a primeira é vazia e a segunda tem dois números
        elif row_list[0] == "" and len(row_list) > 1 and re.match(r'^\d+\s+\d+$', row_list[1]):
            parts = row_list[1].split()
            row_list = [parts[0], parts[1]] + row_list[2:]
        new_rows.append(row_list)

    # Garantir que o número de colunas do DataFrame resultante corresponda ao esperado
    return pd.DataFrame(new_rows)

def find_header_and_data_start(df, expected_headers, max_rows_to_check=15):
    """
    Procura o cabeçalho no DataFrame, ignorando colunas vazias à direita que podem
    ser geradas na extração do PDF.
    """
    for i in range(min(max_rows_to_check, len(df))):
        # Pega os valores da linha e remove espaços
        row_vals = [str(x).strip() for x in df.iloc[i].tolist()]

        # Remove strings vazias do final da lista (comum em extrações 'stream' do Camelot)
        while row_vals and row_vals[-1] == "":
            row_vals.pop()

        row_tuple = tuple(row_vals)

        for h in expected_headers:
            # Compara a tupla encontrada com o cabeçalho esperado
            if row_tuple == h:
                return i, h
    return -1, None

def normalize_dataframe(df_raw, detected_header_tuple, header_row_index):
    """
    Normaliza o DataFrame a partir do cabeçalho detectado.
    """
    if header_row_index == -1: return pd.DataFrame()

    # 1. Corta o DataFrame a partir do cabeçalho
    df_norm = df_raw.iloc[header_row_index:].reset_index(drop=True)

    # 2. Ajusta as colunas para bater com o tamanho do cabeçalho detectado
    num_cols_expected = len(detected_header_tuple)
    df_norm = df_norm.iloc[:, :num_cols_expected]

    # Define nomes temporários para as colunas
    df_norm.columns = [f"col_{i}" for i in range(df_norm.shape[1])]

    # 3. Corrige linhas deslocadas
    df_norm = fix_shifted_rows(df_norm)

    # Redefine colunas após fix_shifted_rows para garantir o tamanho correto
    df_norm = df_norm.iloc[:, :num_cols_expected]

    h1 = ('Coloc.', 'Num.', 'Nome', 'Sx.', 'Idd.', 'Faixa', 'Cl.Fx.', 'Equipe', 'Tempo', 'Liquido')
    h2 = ('Coloc.', 'Numero', 'Nome', 'Id.', 'Fx.Et.', 'Cl.Fx.', 'Sx.', 'C.', 'Equipe', 'Tempo')

    mapping = {
        h1: ['colocacao', 'numero', 'nome', 'sexo', 'idade', 'faixa', 'class_fx', 'equipe', 'tempo', 'liquido'],
        h2: ['colocacao', 'numero', 'nome', 'idade', 'faixa', 'class_fx', 'sexo', 'c', 'equipe', 'tempo']
    }

    df_norm.columns = mapping.get(detected_header_tuple, [f"col_{i}" for i in range(df_norm.shape[1])])

    # 4. Limpeza final
    header_vals = ['Coloc.', 'Num.', 'Numero']
    df_norm = df_norm[~df_norm['colocacao'].isin(header_vals)].reset_index(drop=True)

    # Remove linhas onde o nome está vazio
    if 'nome' in df_norm.columns:
        df_norm = df_norm.replace(r'^\s*$', pd.NA, regex=True).dropna(subset=['nome'], how='all').reset_index(drop=True)

    return df_norm

def extrair_metadados_from_key(key_pdf):
    filename = os.path.basename(key_pdf)

    # 1. Gênero
    genero = "NA"
    if "FEMININO" in filename.upper(): genero = "FEMININO"
    elif "MASCULINO" in filename.upper(): genero = "MASCULINO"

    # 2. Distância
    dist_match = re.search(r'(\d+)\s*KM', filename, re.IGNORECASE)
    distancia = dist_match.group(1) if dist_match else "NA"

    # 3. Data
    data = "NA"
    data_match = re.search(r'(\d{4}-\d{2}-\d{2})', filename)
    if data_match: data = data_match.group(1)

    # 4. Nome da Corrida
    base = filename.rsplit('.', 1)[0].lower()
    nome_corrida = "Desconhecida"

    if "bonissima" in base: nome_corrida = "Boníssima Run"
    elif "supermercado_bh" in base: nome_corrida = "Supermercado BH"
    elif "o_tempo" in base: nome_corrida = "Corrida O Tempo"
    elif "park_run" in base: nome_corrida = "Park Run"
    elif data != "NA":
        parts = re.split(r'\d{4}-\d{2}-\d{2}_', filename)
        if len(parts) > 1:
            nome_part = re.split(r'(_GERAL|_FEMININO|_MASCULINO|_\d+KM)', parts[1], flags=re.IGNORECASE)[0]
            nome_corrida = nome_part.replace('_', ' ').strip().title()
    else:
        clean = re.sub(r'^-?\d+-', '', base)
        clean = re.sub(r'(geral|feminino|masculino|\d+km).*', '', clean, flags=re.IGNORECASE)
        nome_corrida = clean.replace('_', ' ').replace('-', ' ').strip().title()

    return {
        'data': data,
        'corrida': nome_corrida,
        'genero': genero,
        'distancia_km': distancia
    }

def processar_pdf_para_parquet(key_pdf, bucket_raw, bucket_silver):
    """
    Função principal ajustada.
    """
    obj = s3_client.get_object(Bucket=bucket_raw, Key=key_pdf)
    pdf_bytes = obj['Body'].read()

    h1 = ('Coloc.', 'Num.', 'Nome', 'Sx.', 'Idd.', 'Faixa', 'Cl.Fx.', 'Equipe', 'Tempo', 'Liquido')
    h2 = ('Coloc.', 'Numero', 'Nome', 'Id.', 'Fx.Et.', 'Cl.Fx.', 'Sx.', 'C.', 'Equipe', 'Tempo')

    # Tenta extração por lattice (tabelas com bordas) e depois stream (texto alinhado)
    tables = camelot.read_pdf(io.BytesIO(pdf_bytes), pages='all', flavor='lattice')
    if not tables:
        tables = camelot.read_pdf(io.BytesIO(pdf_bytes), pages='all', flavor='stream')

    if not tables:
        return {'arquivo': key_pdf, 'status': 'erro', 'motivo': 'Sem tabelas'}

    dfs = [t.df for t in tables]

    # O ajuste principal foi na find_header_and_data_start para ser resiliente a colunas vazias
    idx, header_tipo = find_header_and_data_start(dfs[0], [h1, h2])

    if not header_tipo:
        return {'arquivo': key_pdf, 'status': 'erro', 'motivo': 'Cabeçalho desconhecido'}

    df_full = pd.concat(dfs, ignore_index=True)
    df_final = normalize_dataframe(df_full, header_tipo, idx)

    # Aplica Metadados
    meta = extrair_metadados_from_key(key_pdf)
    df_final['corrida'] = meta['corrida']
    df_final['genero'] = meta['genero']
    df_final['distancia_km'] = meta['distancia_km']
    df_final['arquivo'] = os.path.basename(key_pdf)

    # Tipagem e Metadados Adicionais (Mantendo chamadas originais)
    df_final = add_metadados(df_final, key_pdf)
    df_final = coerce_types(df_final)

    # Exportação
    file_name = os.path.basename(key_pdf).replace(".pdf", ".parquet")
    data_part = meta.get('data', 'sem_data')
    path_out = f"TBH Esportes/corridas_2026_silver/data={data_part}/{file_name}"

    salvar_parquet_s3(df_final, bucket_silver, path_out)
    return {'arquivo': key_pdf, 'status': 'ok', 'linhas': len(df_final)}


In [ ]:
bucket = S3_BUCKET_RAW
prefix = "TBH Esportes/corridas_2026_completo_final/"

# Lista todos os arquivos do prefixo
pdf_keys = listar_arquivos_s3(s3_client, bucket, prefix)

print(f"Total de arquivos encontrados: {len(pdf_keys)}")
for k in pdf_keys:
    print(f" - {k}")

resultados = []

for key in tqdm(pdf_keys, desc='Processando todos os arquivos'):

    # ignora possíveis "pastas"
    if key.endswith('/'):
        continue

    try:
        info = processar_pdf_para_parquet(
            key_pdf=key,
            bucket_raw=S3_BUCKET_RAW,
            bucket_silver=S3_BUCKET_SILVER
        )
        resultados.append(info)

    except Exception as e:
        print(f"Erro ao processar {key}: {e}")
        resultados.append({
            "arquivo": key,
            "status": "erro",
            "mensagem": str(e)
        })

df_resultados = pd.DataFrame(resultados)

if not df_resultados.empty:
    display(df_resultados)
    print("\nResumo do processamento:")
    print(df_resultados['status'].value_counts(dropna=False))
else:
    print("\nNenhum arquivo encontrado no prefixo.")


In [ ]:
#22/05 - PARA TESTAR ARQUIVOS ESPECIFICOS

bucket = S3_BUCKET_RAW
prefix = "TBH Esportes/corridas_2026_completo_final/"

# Lista todos os arquivos do prefixo
pdf_keys_all = listar_arquivos_s3(s3_client, bucket, prefix)

# FILTRO: Seleciona apenas os arquivos que você deseja testar
# O filtro busca por qualquer parte do nome para ser mais flexível
arquivos_teste = [
    "2026-04-12_Corrida_Farid",
    #"2026-03-01_Boníssima_Run"

]

pdf_keys = [
    key for key in pdf_keys_all
    if any(nome_teste.lower() in key.lower() for nome_teste in arquivos_teste)
]

print(f"Arquivos encontrados para teste: {len(pdf_keys)}")
for k in pdf_keys: print(f" - {k}")

resultados = []
# Agora o loop rodará apenas para os arquivos filtrados
for key in tqdm(pdf_keys, desc='Testando extração específica'):
    info = processar_pdf_para_parquet(
        key_pdf=key,
        bucket_raw=S3_BUCKET_RAW,
        bucket_silver=S3_BUCKET_SILVER
    )
    resultados.append(info)


df_resultados = pd.DataFrame(resultados)
if not df_resultados.empty:
    display(df_resultados)
    print("\nResumo do Teste:")
    print(df_resultados['status'].value_counts(dropna=False))
else:
    print("\nNenhum arquivo encontrado com os nomes especificados. Verifique o prefixo ou os nomes de teste.")
